In [2]:
import pandas as pd
import numpy as np

In [6]:

# Instellingen om te tweaken
# Aantal aanbevelingen; bepaalt ook de K in Precision@K en NDCG@K.
TOP_K = 3
# Maximumaantal meest vergelijkbare studenten dat voor voorspellingen wordt gebruikt.
N_NEIGHBORS = 17
# Minimumaantal gedeelde materialen waarop een studentcorrelatie wordt berekend.
MIN_COMMON_ITEMS = 1
# Minimumaantal gekozen buren dat een materiaal moet hebben beoordeeld.
MIN_ITEM_SUPPORT = 1
# Vanaf deze intscore telt een achtergehouden materiaal als relevant.
RELEVANCE_THRESHOLD = 4
# Deel van de interactieparen per student dat als testdata wordt achtergehouden.
TEST_FRACTION = 0.3
# Verschillende seeds voor herhaalde, reproduceerbare train/test-splits.
# Gebruik dezelfde lijst wanneer je verschillende instellingen vergelijkt.
RANDOM_STATES = [11, 21, 31, 41, 51, 61, 71, 81, 91, 1]
# True gebruikt alleen positief gecorreleerde buren; False laat ook andere correlaties toe.
ONLY_POSITIVE_NEIGHBORS = False
# Kies componenten voor de interactiescore door regels toe te voegen of weg te laten.
# De gewichten geven het relatieve belang; ze worden automatisch genormaliseerd.
SCORE_WEIGHTS = {
    'leesprc': 0.1,      # Afgeleid van leespercentage (omgezet naar een 0-5-schaal).
    # 'inttype': 0.25,      # Afgeleid van interactietype via eval_type_scores hieronder.
    'beoordeling': 1,  # De beoordeling uit de interactiedata.
    # 'leestijd_min': 0.15,  # Leestijd lineair geschaald naar 0-5.
}
if 'user_tuned_settings' in globals():
    N_NEIGHBORS = user_tuned_settings['N_NEIGHBORS']
    MIN_COMMON_ITEMS = user_tuned_settings['MIN_COMMON_ITEMS']
    MIN_ITEM_SUPPORT = user_tuned_settings['MIN_ITEM_SUPPORT']
    ONLY_POSITIVE_NEIGHBORS = user_tuned_settings['ONLY_POSITIVE_NEIGHBORS']
    SCORE_WEIGHTS = dict(globals().get('user_tuned_score_weights', SCORE_WEIGHTS))

# Lees de opgeschoonde data en bereken de interactiescore, onafhankelijk van de baselinecellen.
eval_raw = pd.read_csv('./dataset/cleaned_Interacties.csv')

eval_type_scores = {
    'geliket': 5.0,
    'opgeslagen': 4.0,
    'gelezen': 3.0,
    'bekeken': 2.0,
    'niet_afgemaakt': 1.0,
    'afgewezen': 1.0,
}
eval_unknown_types = (
    set(eval_raw['interactietype'].dropna().unique()) - set(eval_type_scores)
)
if eval_unknown_types:
    raise ValueError('Onbekende interactietypes: {}'.format(sorted(eval_unknown_types)))

# Schaal leestijd lineair naar 0-5 zodat deze vergelijkbaar is met de andere componenten.
eval_reading_time = pd.to_numeric(eval_raw['leestijd_min'], errors='coerce')
eval_reading_time_min = eval_reading_time.min()
eval_reading_time_max = eval_reading_time.max()
if pd.isna(eval_reading_time_min) or pd.isna(eval_reading_time_max):
    raise ValueError('Geen geldige waarden gevonden voor leestijd_min.')
if eval_reading_time_max == eval_reading_time_min:
    raise ValueError('leestijd_min kan niet lineair worden geschaald als alle waarden gelijk zijn.')
eval_reading_time_score = (
    (eval_reading_time - eval_reading_time_min)
    / (eval_reading_time_max - eval_reading_time_min)
    * 5
)

eval_all_inputs = pd.DataFrame({
    'leesprc': (np.ceil(eval_raw['leespercentage'] / 20).clip(1, 5)),
    'inttype': eval_raw['interactietype'].map(eval_type_scores),
    'beoordeling': pd.to_numeric(eval_raw['beoordeling'], errors='coerce'),
    'leestijd_min': eval_reading_time_score,
})
if not SCORE_WEIGHTS:
    raise ValueError('SCORE_WEIGHTS moet minstens één scorecomponent bevatten.')
eval_unknown_components = set(SCORE_WEIGHTS) - set(eval_all_inputs.columns)
if eval_unknown_components:
    raise ValueError('Onbekende scorecomponenten: {}'.format(sorted(eval_unknown_components)))
if any(weight <= 0 for weight in SCORE_WEIGHTS.values()):
    raise ValueError('Alle gekozen scoregewichten moeten groter zijn dan 0.')
eval_inputs = eval_all_inputs[list(SCORE_WEIGHTS)]
eval_weights = pd.Series(SCORE_WEIGHTS)
eval_weighted_sum = eval_inputs.mul(eval_weights, axis='columns').sum(axis=1, min_count=1)
eval_available_weight = eval_inputs.notna().mul(eval_weights, axis='columns').sum(axis=1)
eval_raw['intscore'] = eval_weighted_sum / eval_available_weight

# Gemiddelde score per student-materiaalpaar; voorkom overlap tussen train en test.
eval_pairs = (
    eval_raw.groupby(['student_id', 'cataloguscode'], as_index=False)['intscore']
    .mean()
    .dropna(subset=['intscore'])
    .reset_index(drop=True)
)

# Voer voor elke seed de volledige split, aanbevelingen en evaluatie opnieuw uit.
def evaluate_one_split(eval_random_state):
    eval_rng = np.random.default_rng(eval_random_state)
    eval_test_indices = []
    for _, eval_user_pairs in eval_pairs.groupby('student_id', sort=False):
        if len(eval_user_pairs) < 2:
            continue
        eval_n_test = max(1, round(len(eval_user_pairs) * TEST_FRACTION))
        eval_n_test = min(eval_n_test, len(eval_user_pairs) - 1)
        eval_seed = int(eval_rng.integers(0, 2**32 - 1))
        eval_test_indices.extend(
            eval_user_pairs.sample(n=eval_n_test, random_state=eval_seed).index
        )

    if not eval_test_indices:
        raise ValueError('Niet genoeg student-materiaalparen voor de train/test-split.')

    eval_test = eval_pairs.loc[eval_test_indices].copy()
    eval_train = eval_pairs.drop(index=eval_test_indices).copy()
    eval_students = eval_pairs['student_id'].unique()
    eval_items = eval_pairs['cataloguscode'].unique()
    eval_train_matrix = (
        eval_train.pivot(index='student_id', columns='cataloguscode', values='intscore')
        .reindex(index=eval_students, columns=eval_items)
    )
    eval_similarity = eval_train_matrix.T.corr(min_periods=MIN_COMMON_ITEMS)

    def predict_scores_for_split(user_id):
        eval_neighbors = eval_similarity.loc[user_id].drop(index=user_id).dropna()
        if ONLY_POSITIVE_NEIGHBORS:
            eval_neighbors = eval_neighbors[eval_neighbors > 0]
        eval_neighbors = eval_neighbors.nlargest(N_NEIGHBORS)
        if eval_neighbors.empty:
            return pd.Series(dtype='float64')

        eval_neighbor_scores = eval_train_matrix.loc[eval_neighbors.index]
        eval_weighted_sum = eval_neighbor_scores.mul(
            eval_neighbors, axis=0
        ).sum(axis=0, min_count=1)
        eval_weight_sum = eval_neighbor_scores.notna().mul(
            eval_neighbors, axis=0
        ).sum(axis=0)
        eval_predicted = eval_weighted_sum / eval_weight_sum
        eval_support = eval_neighbor_scores.notna().sum(axis=0)
        eval_unseen = eval_train_matrix.loc[user_id].isna()
        eval_candidates = eval_unseen & (eval_support >= MIN_ITEM_SUPPORT)
        return eval_predicted[eval_candidates]

    eval_test_users = eval_test['student_id'].unique()
    eval_predicted_by_user = {
        eval_user_id: predict_scores_for_split(eval_user_id)
        for eval_user_id in eval_test_users
    }
    eval_recs_by_user = {
        eval_user_id: eval_scores.nlargest(TOP_K).index.tolist()
        for eval_user_id, eval_scores in eval_predicted_by_user.items()
    }
    eval_relevant_by_user = (
        eval_test.loc[eval_test['intscore'] >= RELEVANCE_THRESHOLD]
        .groupby('student_id')['cataloguscode']
        .apply(set)
        .to_dict()
    )
    eval_actual_scores = []
    eval_predicted_scores = []
    for eval_row in eval_test.itertuples(index=False):
        eval_prediction = eval_predicted_by_user[eval_row.student_id].get(
            eval_row.cataloguscode, np.nan
        )
        if pd.notna(eval_prediction):
            eval_actual_scores.append(eval_row.intscore)
            eval_predicted_scores.append(eval_prediction)
    if eval_predicted_scores:
        eval_errors = np.asarray(eval_actual_scores) - np.asarray(eval_predicted_scores)
        eval_mae = np.mean(np.abs(eval_errors))
        eval_rmse = np.sqrt(np.mean(np.square(eval_errors)))
        eval_train_mean = eval_train['intscore'].mean()
        eval_baseline_errors = np.asarray(eval_actual_scores) - eval_train_mean
        eval_baseline_mae = np.mean(np.abs(eval_baseline_errors))
        eval_baseline_rmse = np.sqrt(np.mean(np.square(eval_baseline_errors)))
    else:
        eval_mae = np.nan
        eval_rmse = np.nan
        eval_baseline_mae = np.nan
        eval_baseline_rmse = np.nan

    eval_precision_values = []
    eval_ndcg_values = []
    for eval_user_id, eval_relevant in eval_relevant_by_user.items():
        eval_recs = eval_recs_by_user[eval_user_id]
        eval_hits = [item in eval_relevant for item in eval_recs]
        eval_precision_values.append(sum(eval_hits) / TOP_K)
        eval_dcg = sum(
            1 / np.log2(rank + 1)
            for rank, is_relevant in enumerate(eval_hits, start=1)
            if is_relevant
        )
        eval_ideal_hits = min(TOP_K, len(eval_relevant))
        eval_idcg = sum(
            1 / np.log2(rank + 1)
            for rank in range(1, eval_ideal_hits + 1)
        )
        eval_ndcg_values.append(eval_dcg / eval_idcg if eval_idcg else 0.0)

    eval_candidate_precision_values = []
    eval_candidate_ndcg_values = []
    for eval_user_id, eval_relevant in eval_relevant_by_user.items():
        eval_candidate_test_items = eval_test.loc[
            eval_test['student_id'] == eval_user_id, 'cataloguscode'
        ].tolist()
        eval_candidate_scores = eval_predicted_by_user[eval_user_id].reindex(
            eval_candidate_test_items
        ).dropna()
        eval_candidate_recs = eval_candidate_scores.nlargest(TOP_K).index.tolist()
        eval_candidate_hits = [
            eval_item in eval_relevant for eval_item in eval_candidate_recs
        ]
        eval_candidate_precision_values.append(sum(eval_candidate_hits) / TOP_K)
        eval_candidate_dcg = sum(
            1 / np.log2(rank + 1)
            for rank, is_relevant in enumerate(eval_candidate_hits, start=1)
            if is_relevant
        )
        eval_candidate_ideal_hits = min(TOP_K, len(eval_relevant))
        eval_candidate_idcg = sum(
            1 / np.log2(rank + 1)
            for rank in range(1, eval_candidate_ideal_hits + 1)
        )
        eval_candidate_ndcg_values.append(
            eval_candidate_dcg / eval_candidate_idcg
            if eval_candidate_idcg else 0.0
        )

    eval_known_items_by_user = (
        eval_test.groupby('student_id')['cataloguscode'].apply(set).to_dict()
    )
    eval_judged_precision_values = []
    eval_judged_shares = []
    eval_top_k_coverages = []
    eval_judged_user_count = 0
    for eval_user_id, eval_known_items in eval_known_items_by_user.items():
        eval_recs = eval_recs_by_user[eval_user_id]
        eval_judged_recs = [item for item in eval_recs if item in eval_known_items]
        eval_relevant = eval_relevant_by_user.get(eval_user_id, set())
        eval_judged_hits = sum(item in eval_relevant for item in eval_judged_recs)
        if eval_judged_recs:
            eval_judged_user_count += 1
            eval_judged_precision_values.append(
                eval_judged_hits / len(eval_judged_recs)
            )
        else:
            eval_judged_precision_values.append(np.nan)
        eval_judged_shares.append(
            len(eval_judged_recs) / len(eval_recs) if eval_recs else np.nan
        )
        eval_top_k_coverages.append(len(eval_judged_recs) / TOP_K)

    eval_total_pairs = len(eval_pairs)
    return {
        'random_state': eval_random_state,
        'train_percent': 100 * len(eval_train) / eval_total_pairs,
        'test_percent': 100 * len(eval_test) / eval_total_pairs,
        'students_with_relevant_test': len(eval_precision_values),
        'students_with_judged_recommendations': eval_judged_user_count,
        'average_recommendations_returned': pd.Series(
            [len(recs) for recs in eval_recs_by_user.values()], dtype='float64'
        ).mean(),
        'Precision@K': pd.Series(eval_precision_values, dtype='float64').mean(),
        'NDCG@K': pd.Series(eval_ndcg_values, dtype='float64').mean(),
        'Candidate Ranking Precision@K': pd.Series(
            eval_candidate_precision_values, dtype='float64'
        ).mean(),
        'Candidate Ranking NDCG@K': pd.Series(
            eval_candidate_ndcg_values, dtype='float64'
        ).mean(),
        'Judged-only Precision@K': pd.Series(
            eval_judged_precision_values, dtype='float64'
        ).mean(),
        'judged_share_of_recommendations': pd.Series(
            eval_judged_shares, dtype='float64'
        ).mean(),
        'judged_top_k_coverage': pd.Series(
            eval_top_k_coverages, dtype='float64'
        ).mean(),
        'MAE': eval_mae,
        'RMSE': eval_rmse,
        'Global-mean baseline MAE': eval_baseline_mae,
        'Global-mean baseline RMSE': eval_baseline_rmse,
        'score_prediction_coverage': len(eval_predicted_scores) / len(eval_test),
        'predicted_test_pairs': len(eval_predicted_scores),
    }

if not RANDOM_STATES:
    raise ValueError('RANDOM_STATES moet minstens ??n seed bevatten.')
if len(set(RANDOM_STATES)) != len(RANDOM_STATES):
    raise ValueError('RANDOM_STATES mag geen dubbele seeds bevatten.')

eval_run_metrics = pd.DataFrame(
    [evaluate_one_split(eval_seed) for eval_seed in RANDOM_STATES]
)

eval_component_sources = {
    'leesprc': 'leespercentage / 20, afgerond',
    'inttype': 'interactietype (gescoord via eval_type_scores)',
    'beoordeling': 'beoordeling',
    'leestijd_min': 'leestijd_min lineair geschaald naar 0-5',
}
print('Gebruikte scorecomponenten en ingestelde relatieve gewichten:')
for eval_component, eval_weight in SCORE_WEIGHTS.items():
    print('  {} <- {}: {}'.format(
        eval_component, eval_component_sources[eval_component], eval_weight
    ))
print('Instellingen (1/2): splits={} | test={:.0f}% | train/test={:.1f}/{:.1f}% | K={}'.format(
    len(RANDOM_STATES), 100 * TEST_FRACTION,
    eval_run_metrics['train_percent'].mean(), eval_run_metrics['test_percent'].mean(),
    TOP_K
))
print('Instellingen (2/2): buren={} | overlap={} | support={} | drempel={} | seeds={}'.format(
    N_NEIGHBORS, MIN_COMMON_ITEMS, MIN_ITEM_SUPPORT, RELEVANCE_THRESHOLD, RANDOM_STATES
))
print('Gemiddelde +/- standaardafwijking tussen splits:')

def print_run_summary(eval_label, eval_column):
    eval_values = eval_run_metrics[eval_column].dropna()
    if eval_values.empty:
        print('{}: geen geldige resultaten'.format(eval_label))
        return
    print('{}: {:.4f} +/- {:.4f}'.format(
        eval_label, eval_values.mean(), eval_values.std(ddof=0)
    ))

for eval_label, eval_column in [
    ('Full-catalog Precision@{}'.format(TOP_K), 'Precision@K'),
    ('Full-catalog NDCG@{}'.format(TOP_K), 'NDCG@K'),
    ('Candidate Ranking Precision@{}'.format(TOP_K), 'Candidate Ranking Precision@K'),
    ('Candidate Ranking NDCG@{}'.format(TOP_K), 'Candidate Ranking NDCG@K'),
    ('Judged-only Precision@{}'.format(TOP_K), 'Judged-only Precision@K'),
    ('Aandeel beoordeeld in aanbevelingen', 'judged_share_of_recommendations'),
    ('Studenten met relevante testitems', 'students_with_relevant_test'),
    ('Studenten met beoordeelde aanbevelingen', 'students_with_judged_recommendations'),
]:
    print_run_summary(eval_label, eval_column)


Gebruikte scorecomponenten en ingestelde relatieve gewichten:
  leesprc <- leespercentage / 20, afgerond: 0.1
  beoordeling <- beoordeling: 1
Instellingen (1/2): splits=10 | test=30% | train/test=70.0/30.0% | K=3
Instellingen (2/2): buren=17 | overlap=1 | support=1 | drempel=4 | seeds=[11, 21, 31, 41, 51, 61, 71, 81, 91, 1]
Gemiddelde +/- standaardafwijking tussen splits:
Full-catalog Precision@3: 0.0202 +/- 0.0051
Full-catalog NDCG@3: 0.0244 +/- 0.0081
Candidate Ranking Precision@3: 0.4245 +/- 0.0290
Candidate Ranking NDCG@3: 0.5613 +/- 0.0369
Judged-only Precision@3: 0.4914 +/- 0.1256
Aandeel beoordeeld in aanbevelingen: 0.0377 +/- 0.0092
Studenten met relevante testitems: 89.3000 +/- 2.1471
Studenten met beoordeelde aanbevelingen: 11.1000 +/- 2.8443


In [5]:
# Run cel 2 eerst. De baseline voorspelt voor elk testpaar het gemiddelde van de train-scores.
# De vergelijking gebruikt alleen testparen waarvoor collaborative filtering een score kon voorspellen.
eval_comparison = pd.DataFrame({
    'Model': ['Collaborative filtering', 'Gemiddelde train-score baseline'],
    'MAE gemiddeld': [
        eval_run_metrics['MAE'].mean(),
        eval_run_metrics['Global-mean baseline MAE'].mean(),
    ],
    'MAE SD': [
        eval_run_metrics['MAE'].std(ddof=0),
        eval_run_metrics['Global-mean baseline MAE'].std(ddof=0),
    ],
    'RMSE gemiddeld': [
        eval_run_metrics['RMSE'].mean(),
        eval_run_metrics['Global-mean baseline RMSE'].mean(),
    ],
    'RMSE SD': [
        eval_run_metrics['RMSE'].std(ddof=0),
        eval_run_metrics['Global-mean baseline RMSE'].std(ddof=0),
    ],
})
print('Fout op dezelfde voorspelbare testparen (lager is beter):')
print(eval_comparison.round(4).to_string(index=False))
print('Dekking collaborative filtering: {:.1f}%'.format(
    100 * eval_run_metrics['score_prediction_coverage'].mean()
))

Fout op dezelfde voorspelbare testparen (lager is beter):
                          Model  MAE gemiddeld  MAE SD  RMSE gemiddeld  RMSE SD
        Collaborative filtering         0.8757  0.0485          1.1174   0.0542
Gemiddelde train-score baseline         0.7632  0.0346          0.9349   0.0294
Dekking collaborative filtering: 28.8%


In [ ]:
# Fout op dezelfde voorspelbare testparen (lager is beter):
#                           Model  MAE gemiddeld  MAE SD  RMSE gemiddeld  RMSE SD
#         Collaborative filtering         0.9179  0.0487          1.1763   0.0579
# Gemiddelde train-score baseline         0.8061  0.0496          0.9848   0.0465
# Dekking collaborative filtering: 28.6%

In [6]:
# Item-based collaborative filtering: vergelijk items op basis van train-interacties.
# Scores worden per student gecentreerd; alles wordt per split alleen op train-data geleerd.
MIN_COMMON_STUDENTS = 3


def evaluate_item_based_split(eval_random_state):
    eval_item_rng = np.random.default_rng(eval_random_state)
    eval_item_test_indices = []
    for _, eval_item_user_pairs in eval_pairs.groupby('student_id', sort=False):
        if len(eval_item_user_pairs) < 2:
            continue
        eval_item_n_test = max(1, round(len(eval_item_user_pairs) * TEST_FRACTION))
        eval_item_n_test = min(eval_item_n_test, len(eval_item_user_pairs) - 1)
        eval_item_seed = int(eval_item_rng.integers(0, 2**32 - 1))
        eval_item_test_indices.extend(
            eval_item_user_pairs.sample(
                n=eval_item_n_test, random_state=eval_item_seed
            ).index
        )

    if not eval_item_test_indices:
        raise ValueError('Niet genoeg student-materiaalparen voor de train/test-split.')

    eval_item_test = eval_pairs.loc[eval_item_test_indices].copy()
    eval_item_train = eval_pairs.drop(index=eval_item_test_indices).copy()
    eval_item_students = eval_pairs['student_id'].unique()
    eval_item_ids = eval_pairs['cataloguscode'].unique()
    eval_item_train_matrix = (
        eval_item_train.pivot(
            index='student_id', columns='cataloguscode', values='intscore'
        )
        .reindex(index=eval_item_students, columns=eval_item_ids)
    )

    eval_item_user_means = eval_item_train_matrix.mean(axis=1)
    eval_item_centered_matrix = eval_item_train_matrix.sub(
        eval_item_user_means, axis='index'
    )
    eval_item_similarities = eval_item_centered_matrix.corr(
        min_periods=MIN_COMMON_STUDENTS
    )
    eval_item_support = eval_item_train_matrix.notna().sum(axis=0)

    def predict_item_scores(eval_item_user_id):
        eval_item_history = eval_item_train_matrix.loc[eval_item_user_id].dropna()
        if eval_item_history.empty:
            return pd.Series(dtype='float64')

        eval_item_centered_history = eval_item_centered_matrix.loc[
            eval_item_user_id
        ].dropna()
        eval_item_unseen = eval_item_train_matrix.loc[eval_item_user_id].isna()
        eval_item_candidates = eval_item_unseen & (
            eval_item_support >= MIN_ITEM_SUPPORT
        )
        eval_item_predictions = {}

        for eval_item_candidate in eval_item_train_matrix.columns[eval_item_candidates]:
            eval_item_neighbors = (
                eval_item_similarities.loc[eval_item_candidate]
                .reindex(eval_item_history.index)
                .dropna()
            )
            if ONLY_POSITIVE_NEIGHBORS:
                eval_item_neighbors = eval_item_neighbors[eval_item_neighbors > 0]
            eval_item_neighbors = eval_item_neighbors.nlargest(N_NEIGHBORS)
            if eval_item_neighbors.empty:
                continue

            eval_item_weight_total = eval_item_neighbors.abs().sum()
            if eval_item_weight_total == 0:
                continue
            eval_item_deviations = eval_item_centered_history.reindex(
                eval_item_neighbors.index
            )
            eval_item_predictions[eval_item_candidate] = (
                eval_item_user_means.loc[eval_item_user_id]
                + (eval_item_neighbors * eval_item_deviations).sum()
                / eval_item_weight_total
            )

        return pd.Series(eval_item_predictions, dtype='float64')

    eval_item_test_users = eval_item_test['student_id'].unique()
    eval_item_predictions_by_user = {
        eval_item_user_id: predict_item_scores(eval_item_user_id)
        for eval_item_user_id in eval_item_test_users
    }

    eval_item_actual_scores = []
    eval_item_predicted_scores = []
    for eval_item_row in eval_item_test.itertuples(index=False):
        eval_item_prediction = eval_item_predictions_by_user[
            eval_item_row.student_id
        ].get(eval_item_row.cataloguscode, np.nan)
        if pd.notna(eval_item_prediction):
            eval_item_actual_scores.append(eval_item_row.intscore)
            eval_item_predicted_scores.append(eval_item_prediction)

    if eval_item_predicted_scores:
        eval_item_errors = (
            np.asarray(eval_item_actual_scores)
            - np.asarray(eval_item_predicted_scores)
        )
        eval_item_mae = np.mean(np.abs(eval_item_errors))
        eval_item_rmse = np.sqrt(np.mean(np.square(eval_item_errors)))
        eval_item_train_mean = eval_item_train['intscore'].mean()
        eval_item_baseline_errors = (
            np.asarray(eval_item_actual_scores) - eval_item_train_mean
        )
        eval_item_baseline_mae = np.mean(np.abs(eval_item_baseline_errors))
        eval_item_baseline_rmse = np.sqrt(
            np.mean(np.square(eval_item_baseline_errors))
        )
    else:
        eval_item_mae = np.nan
        eval_item_rmse = np.nan
        eval_item_baseline_mae = np.nan
        eval_item_baseline_rmse = np.nan

    eval_item_relevant_by_user = (
        eval_item_test.loc[
            eval_item_test['intscore'] >= RELEVANCE_THRESHOLD
        ]
        .groupby('student_id')['cataloguscode']
        .apply(set)
        .to_dict()
    )
    eval_item_precision = []
    eval_item_ndcg = []
    for eval_item_user_id, eval_item_relevant in eval_item_relevant_by_user.items():
        eval_item_test_candidates = eval_item_test.loc[
            eval_item_test['student_id'] == eval_item_user_id, 'cataloguscode'
        ].tolist()
        eval_item_candidate_scores = eval_item_predictions_by_user[
            eval_item_user_id
        ].reindex(eval_item_test_candidates).dropna()
        eval_item_recs = eval_item_candidate_scores.nlargest(TOP_K).index.tolist()
        eval_item_hits = [
            eval_item_id in eval_item_relevant for eval_item_id in eval_item_recs
        ]
        eval_item_precision.append(sum(eval_item_hits) / TOP_K)
        eval_item_dcg = sum(
            1 / np.log2(eval_item_rank + 1)
            for eval_item_rank, eval_item_hit in enumerate(eval_item_hits, start=1)
            if eval_item_hit
        )
        eval_item_ideal_count = min(TOP_K, len(eval_item_relevant))
        eval_item_idcg = sum(
            1 / np.log2(eval_item_rank + 1)
            for eval_item_rank in range(1, eval_item_ideal_count + 1)
        )
        eval_item_ndcg.append(
            eval_item_dcg / eval_item_idcg if eval_item_idcg else 0.0
        )

    return {
        'MAE': eval_item_mae,
        'RMSE': eval_item_rmse,
        'Train-mean baseline MAE': eval_item_baseline_mae,
        'Train-mean baseline RMSE': eval_item_baseline_rmse,
        'Precision@K': pd.Series(eval_item_precision, dtype='float64').mean(),
        'NDCG@K': pd.Series(eval_item_ndcg, dtype='float64').mean(),
        'test_pair_coverage': len(eval_item_predicted_scores) / len(eval_item_test),
    }


eval_item_run_metrics = pd.DataFrame(
    [evaluate_item_based_split(eval_seed) for eval_seed in RANDOM_STATES]
)

eval_item_comparison = pd.DataFrame({
    'Model': [
        'Item-based collaborative filtering',
        'Gemiddelde train-score baseline (zelfde voorspelbare testparen)',
    ],
    'MAE gemiddeld': [
        eval_item_run_metrics['MAE'].mean(),
        eval_item_run_metrics['Train-mean baseline MAE'].mean(),
    ],
    'MAE SD': [
        eval_item_run_metrics['MAE'].std(ddof=0),
        eval_item_run_metrics['Train-mean baseline MAE'].std(ddof=0),
    ],
    'RMSE gemiddeld': [
        eval_item_run_metrics['RMSE'].mean(),
        eval_item_run_metrics['Train-mean baseline RMSE'].mean(),
    ],
    'RMSE SD': [
        eval_item_run_metrics['RMSE'].std(ddof=0),
        eval_item_run_metrics['Train-mean baseline RMSE'].std(ddof=0),
    ],
})

print('Item-based CF: scorevoorspelling op testparen (lager is beter):')
print(eval_item_comparison.round(4).to_string(index=False))
print('Voorspellingsdekking testparen: {:.1f}% +/- {:.1f}%'.format(
    100 * eval_item_run_metrics['test_pair_coverage'].mean(),
    100 * eval_item_run_metrics['test_pair_coverage'].std(ddof=0),
))
print('Candidate Ranking op achtergehouden testitems:')
for eval_item_label, eval_item_column in [
    ('Precision@{}'.format(TOP_K), 'Precision@K'),
    ('NDCG@{}'.format(TOP_K), 'NDCG@K'),
]:
    eval_item_values = eval_item_run_metrics[eval_item_column].dropna()
    if eval_item_values.empty:
        print('{}: geen geldige resultaten'.format(eval_item_label))
    else:
        print('{}: {:.4f} +/- {:.4f}'.format(
            eval_item_label,
            eval_item_values.mean(),
            eval_item_values.std(ddof=0),
        ))

Item-based CF: scorevoorspelling op testparen (lager is beter):
                                                          Model  MAE gemiddeld  MAE SD  RMSE gemiddeld  RMSE SD
                             Item-based collaborative filtering         0.9382  0.0419          1.1599   0.0549
Gemiddelde train-score baseline (zelfde voorspelbare testparen)         0.7701  0.0537          0.9390   0.0418
Voorspellingsdekking testparen: 18.7% +/- 1.6%
Candidate Ranking op achtergehouden testitems:
Precision@3: 0.1073 +/- 0.0168
NDCG@3: 0.2576 +/- 0.0302


In [13]:
# Coordinate search voor beide modellen; elke kandidaat gebruikt dezelfde 10 seeds.
# Beoordeling blijft verplicht; de andere scorekolommen en hun gewichten worden meegetuned.
TOP_K = 3
N_NEIGHBORS = 3
MIN_COMMON_ITEMS = 1
MIN_ITEM_SUPPORT = 1
MIN_COMMON_STUDENTS = 3
ONLY_POSITIVE_NEIGHBORS = True
SCORE_WEIGHTS = {'leesprc': 0.15, 'inttype': 0.25, 'beoordeling': 0.75, 'leestijd_min': 0.15}
if len(RANDOM_STATES) != 10 or len(set(RANDOM_STATES)) != 10:
    raise ValueError('Cell 7 vereist precies 10 unieke RANDOM_STATES.')

def tuning_batch(model, evaluator, columns, label):
    frame = pd.DataFrame([evaluator(seed) for seed in RANDOM_STATES])
    precision_col, ndcg_col, baseline_mae_col, baseline_rmse_col = columns
    frame['_delta_mae'] = frame['MAE'] - frame[baseline_mae_col]
    frame['_delta_rmse'] = frame['RMSE'] - frame[baseline_rmse_col]
    summary = {}
    for key, col in [('precision', precision_col), ('ndcg', ndcg_col), ('mae', 'MAE'), ('rmse', 'RMSE'), ('baseline_mae', baseline_mae_col), ('baseline_rmse', baseline_rmse_col), ('delta_mae', '_delta_mae'), ('delta_rmse', '_delta_rmse')]:
        values = frame[col].dropna()
        summary[key] = (values.mean(), values.std(ddof=0)) if len(values) else (np.nan, np.nan)
    p, ps = summary['precision']; n, ns = summary['ndcg']
    m, ms = summary['mae']; r, rs = summary['rmse']
    bm, bms = summary['baseline_mae']; br, brs = summary['baseline_rmse']
    dm, dms = summary['delta_mae']; dr, drs = summary['delta_rmse']
    print('[{}] {} | Precision={:.4f} +/- {:.4f} | NDCG={:.4f} +/- {:.4f}'.format(model, label, p, ps, n, ns))
    print('    MAE={:.4f} +/- {:.4f}; baseline={:.4f} +/- {:.4f}; verschil={:+.4f} +/- {:.4f}'.format(m, ms, bm, bms, dm, dms))
    print('    RMSE={:.4f} +/- {:.4f}; baseline={:.4f} +/- {:.4f}; verschil={:+.4f} +/- {:.4f}'.format(r, rs, br, brs, dr, drs))
    return frame, summary

def ranking_score(summary):
    p, n = summary['precision'][0], summary['ndcg'][0]
    return (p + n) / 2 if np.isfinite(p) and np.isfinite(n) else -np.inf

def ranking_metrics_should_stop(previous, candidate):
    old_p, old_n = previous['precision'][0], previous['ndcg'][0]
    new_p, new_n = candidate['precision'][0], candidate['ndcg'][0]
    if not all(np.isfinite(value) for value in [old_p, old_n, new_p, new_n]):
        return True
    return (
        new_p < old_p or new_n < old_n
        or (np.isclose(new_p, old_p) and np.isclose(new_n, old_n))
    )

def tune_setting(model, evaluator, columns, name, low, high, current_frame, current_summary):
    start = globals()[name]
    print('\nInstelling {} (start={})'.format(name, start))
    options = [(start, current_frame, current_summary)]
    for step, direction in [(1, 'omhoog'), (-1, 'omlaag')]:
        globals()[name] = start
        previous = current_summary
        while True:
            value = globals()[name] + step
            if value < low or value > high:
                print('    Stop {}: grens {} bereikt.'.format(direction, globals()[name]))
                break
            globals()[name] = value
            frame, summary = tuning_batch(model, evaluator, columns, '{} {} -> {}'.format(name, direction, value))
            if ranking_metrics_should_stop(previous, summary):
                print('    Stop {}: een metric daalde, beide bleven gelijk of een metric was ongeldig.'.format(direction))
                break
            options.append((value, frame, summary))
            previous = summary
    best = max(options, key=lambda option: ranking_score(option[2]))
    globals()[name] = best[0]
    print('    Gekozen {}={}; gemiddelde Precision/NDCG={:.4f}'.format(name, best[0], ranking_score(best[2])))
    return best[1], best[2]

def tune_model(model, evaluator, columns, specs):
    print('\n' + '=' * 70 + '\n{}: startmeting (10 seeds)'.format(model))
    frame, summary = tuning_batch(model, evaluator, columns, 'startinstellingen')
    for name, low, high in specs:
        frame, summary = tune_setting(model, evaluator, columns, name, low, high, frame, summary)
    print('{}: gekozen instellingen'.format(model))
    for name, _, _ in specs:
        print('    {}={}'.format(name, globals()[name]))
    tuning_batch(model, evaluator, columns, 'eindresultaat')
    return frame

score_features = ['leesprc', 'inttype', 'leestijd_min']
if 'beoordeling' not in eval_all_inputs.columns or SCORE_WEIGHTS.get('beoordeling', 0) <= 0:
    raise ValueError('Beoordeling moet aanwezig zijn en een positief gewicht hebben.')
score_weight_units = {
    feature: min(9, max(0, int(round(SCORE_WEIGHTS.get(feature, 0) / SCORE_WEIGHTS['beoordeling'] * 10))))
    for feature in score_features
}

def apply_score_weight_units(units):
    global SCORE_WEIGHTS, eval_pairs
    weights = {'beoordeling': 1.0}
    weights.update({feature: units[feature] / 10 for feature in score_features if units[feature] > 0})
    inputs = eval_all_inputs[list(weights)]
    weight_series = pd.Series(weights)
    weighted_sum = inputs.mul(weight_series, axis='columns').sum(axis=1, min_count=1)
    available_weight = inputs.notna().mul(weight_series, axis='columns').sum(axis=1)
    scored = eval_raw[['student_id', 'cataloguscode']].copy()
    scored['intscore'] = weighted_sum / available_weight
    SCORE_WEIGHTS = weights
    eval_pairs = (
        scored.groupby(['student_id', 'cataloguscode'], as_index=False)['intscore']
        .mean().dropna(subset=['intscore']).reset_index(drop=True)
    )

def evaluate_score_weights(units, label):
    apply_score_weight_units(units)
    omitted = [feature for feature in score_features if units[feature] == 0]
    print('\nScoremix {}: gewichten={} | weggelaten={}'.format(label, SCORE_WEIGHTS, omitted))
    user_frame, user_summary = tuning_batch(
        'User-based CF', evaluate_one_split,
        ('Candidate Ranking Precision@K', 'Candidate Ranking NDCG@K',
         'Global-mean baseline MAE', 'Global-mean baseline RMSE'), label,
    )
    item_frame, item_summary = tuning_batch(
        'Item-based CF', evaluate_item_based_split,
        ('Precision@K', 'NDCG@K', 'Train-mean baseline MAE',
         'Train-mean baseline RMSE'), label,
    )
    precision_by_seed = pd.concat([
        user_frame['Candidate Ranking Precision@K'], item_frame['Precision@K'],
    ], axis=1).mean(axis=1).dropna()
    ndcg_by_seed = pd.concat([
        user_frame['Candidate Ranking NDCG@K'], item_frame['NDCG@K'],
    ], axis=1).mean(axis=1).dropna()
    summary = {
        'precision': (precision_by_seed.mean(), precision_by_seed.std(ddof=0)),
        'ndcg': (ndcg_by_seed.mean(), ndcg_by_seed.std(ddof=0)),
    }
    print('    Gecombineerd: Precision={:.4f} +/- {:.4f} | NDCG={:.4f} +/- {:.4f}'.format(
        summary['precision'][0], summary['precision'][1],
        summary['ndcg'][0], summary['ndcg'][1]
    ))
    return summary

from itertools import combinations
print('\n' + '=' * 70 + '\nAlle combinaties van scorekolommen testen; beoordeling blijft verplicht')
score_subsets = [
    subset
    for subset_size in range(len(score_features) + 1)
    for subset in combinations(score_features, subset_size)
]
score_subset_results = []
for subset in score_subsets:
    subset_units = {
        feature: max(1, score_weight_units[feature]) if feature in subset else 0
        for feature in score_features
    }
    subset_label = '+'.join(subset) if subset else 'alleen beoordeling'
    subset_summary = evaluate_score_weights(subset_units, 'kolommen: ' + subset_label)
    score_subset_results.append((subset, subset_units, subset_summary))
best_subset, score_weight_units, score_summary = max(
    score_subset_results, key=lambda result: ranking_score(result[2])
)
apply_score_weight_units(score_weight_units)
print('Beste kolomcombinatie:', list(best_subset) or ['beoordeling'])

print('\nWegingen binnen de beste kolomcombinatie optimaliseren')
for feature in best_subset:
    start = score_weight_units[feature]
    options = [(start, score_summary)]
    for step, direction in [(1, 'omhoog'), (-1, 'omlaag')]:
        score_weight_units[feature] = start
        previous = score_summary
        while 0 <= score_weight_units[feature] + step <= 9:
            value = score_weight_units[feature] + step
            candidate_units = dict(score_weight_units)
            candidate_units[feature] = value
            candidate = evaluate_score_weights(
                candidate_units, '{} {} -> {:.1f}'.format(feature, direction, value / 10)
            )
            if ranking_metrics_should_stop(previous, candidate):
                print('    Stop {}: een metric daalde, beide bleven gelijk of een metric was ongeldig.'.format(direction))
                break
            options.append((value, candidate))
            score_weight_units[feature] = value
            previous = candidate
        score_weight_units[feature] = start
    best_value, score_summary = max(options, key=lambda option: ranking_score(option[1]))
    score_weight_units[feature] = best_value
    apply_score_weight_units(score_weight_units)
    print('    Gekozen gewicht {}={:.1f}'.format(feature, best_value / 10))

print('Eindscoremix:', SCORE_WEIGHTS)
print('Weggelaten scorekolommen:', [feature for feature in score_features if score_weight_units[feature] == 0])

tuning_students = eval_pairs['student_id'].nunique()
tuning_items = eval_pairs['cataloguscode'].nunique()
initial_settings = {'TOP_K': TOP_K, 'N_NEIGHBORS': N_NEIGHBORS, 'MIN_ITEM_SUPPORT': MIN_ITEM_SUPPORT, 'ONLY_POSITIVE_NEIGHBORS': ONLY_POSITIVE_NEIGHBORS, 'MIN_COMMON_ITEMS': MIN_COMMON_ITEMS, 'MIN_COMMON_STUDENTS': MIN_COMMON_STUDENTS}
user_specs = [('N_NEIGHBORS', 1, min(20, max(1, tuning_students - 1))), ('MIN_COMMON_ITEMS', 1, min(10, max(1, tuning_items))), ('MIN_ITEM_SUPPORT', 1, min(10, max(1, tuning_students))), ('ONLY_POSITIVE_NEIGHBORS', 0, 1)]
item_specs = [('N_NEIGHBORS', 1, min(20, max(1, tuning_items - 1))), ('MIN_COMMON_STUDENTS', 1, min(10, max(1, tuning_students))), ('MIN_ITEM_SUPPORT', 1, min(10, max(1, tuning_students))), ('ONLY_POSITIVE_NEIGHBORS', 0, 1)]

eval_run_metrics = tune_model('User-based CF', evaluate_one_split, ('Candidate Ranking Precision@K', 'Candidate Ranking NDCG@K', 'Global-mean baseline MAE', 'Global-mean baseline RMSE'), user_specs)
user_tuned_settings = {name: globals()[name] for name, _, _ in user_specs}
user_tuned_score_weights = dict(SCORE_WEIGHTS)
for name, value in initial_settings.items():
    globals()[name] = value
eval_item_run_metrics = tune_model('Item-based CF', evaluate_item_based_split, ('Precision@K', 'NDCG@K', 'Train-mean baseline MAE', 'Train-mean baseline RMSE'), item_specs)
item_tuned_settings = {name: globals()[name] for name, _, _ in item_specs}
for name, value in user_tuned_settings.items():
    globals()[name] = value
print('\nTuning afgerond.')
print('User-based instellingen:', user_tuned_settings)
print('Item-based instellingen:', item_tuned_settings)
print('Gekozen scorekolommen en gewichten:', SCORE_WEIGHTS)

def print_final_model_metrics(model_name, frame, precision_column, ndcg_column,
                              baseline_mae_column, baseline_rmse_column):
    print('\n{}: uiteindelijke resultaten over {} seeds'.format(model_name, len(frame)))
    for label, column in [
        ('Precision@{}'.format(TOP_K), precision_column),
        ('NDCG@{}'.format(TOP_K), ndcg_column),
        ('MAE', 'MAE'),
        ('RMSE', 'RMSE'),
    ]:
        values = frame[column].dropna()
        if values.empty:
            print('{}: geen geldige resultaten'.format(label))
        else:
            print('{}: {:.4f} +/- {:.4f}'.format(label, values.mean(), values.std(ddof=0)))
    for metric, baseline_column, model_column in [
        ('MAE', baseline_mae_column, 'MAE'),
        ('RMSE', baseline_rmse_column, 'RMSE'),
    ]:
        baseline = frame[baseline_column].dropna()
        difference = (frame[model_column] - frame[baseline_column]).dropna()
        if not baseline.empty and not difference.empty:
            print('{} baseline: {:.4f} +/- {:.4f}; model-baseline: {:+.4f} +/- {:.4f}'.format(
                metric, baseline.mean(), baseline.std(ddof=0),
                difference.mean(), difference.std(ddof=0),
            ))

print_final_model_metrics(
    'User-based CF', eval_run_metrics,
    'Candidate Ranking Precision@K', 'Candidate Ranking NDCG@K',
    'Global-mean baseline MAE', 'Global-mean baseline RMSE',
)
print_final_model_metrics(
    'Item-based CF', eval_item_run_metrics,
    'Precision@K', 'NDCG@K',
    'Train-mean baseline MAE', 'Train-mean baseline RMSE',
)
print('MAE/RMSE-verschil = model minus train-setgemiddelde; negatief is beter.')


Alle combinaties van scorekolommen testen; beoordeling blijft verplicht

Scoremix kolommen: alleen beoordeling: gewichten={'beoordeling': 1.0} | weggelaten=['leesprc', 'inttype', 'leestijd_min']
[User-based CF] kolommen: alleen beoordeling | Precision=0.1248 +/- 0.0243 | NDCG=0.2168 +/- 0.0422
    MAE=0.7767 +/- 0.0834; baseline=0.6754 +/- 0.0458; verschil=+0.1013 +/- 0.0841
    RMSE=1.0411 +/- 0.0832; baseline=0.8350 +/- 0.0393; verschil=+0.2060 +/- 0.0938
[Item-based CF] kolommen: alleen beoordeling | Precision=0.0942 +/- 0.0182 | NDCG=0.1662 +/- 0.0380
    MAE=0.7778 +/- 0.0800; baseline=0.6512 +/- 0.0647; verschil=+0.1266 +/- 0.0750
    RMSE=1.0211 +/- 0.0923; baseline=0.8082 +/- 0.0688; verschil=+0.2129 +/- 0.0733
    Gecombineerd: Precision=0.1095 +/- 0.0145 | NDCG=0.1915 +/- 0.0304

Scoremix kolommen: leesprc: gewichten={'beoordeling': 1.0, 'leesprc': 0.2} | weggelaten=['inttype', 'leestijd_min']
[User-based CF] kolommen: leesprc | Precision=0.1828 +/- 0.0173 | NDCG=0.2816 +/- 

KeyboardInterrupt: 